In [1]:
import timm
import torch
import numpy as np
import torch.nn as nn
import numpy as np
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
import time
model = timm.create_model('vit_base_patch16_224', pretrained=True)

config = resolve_data_config({}, model=model)
transform = create_transform(**config)

In [2]:
file_path = "pre_weights/dummy_input_4_196_768.bin"
with open(file_path, 'rb') as f:
    data = np.fromfile(f, dtype=np.float32).reshape(4, 196, 768)

In [3]:
torch.manual_seed(10)
inputs = torch.randn(4, 196, 768).reshape(4, 14, 14, 3, 16, 16).permute(0, 3, 1, 4, 2, 5).reshape(4, 3, 224, 224)

In [4]:

input = torch.tensor(data, dtype=torch.float16).reshape(4, 14, 14, 3, 16, 16).permute(0, 3, 1, 4, 2, 5).reshape(4, 3, 224, 224)
# inputs = torch.tensor(data, dtype=torch.float32).reshape(4, 14, 14, 3, 16, 16).permute(0, 3, 1, 4, 2, 5).reshape(4, 3, 224, 224)
type(inputs)

torch.Tensor

In [5]:
for i in range(12):
    list(model.blocks.children())[i].attn.flash_attn = False

In [6]:
(dict(model.patch_embed.named_children())['proj'].weight).dtype

torch.float32

In [7]:
device = torch.device('cuda')
start = time.time()
inputs = torch.tensor(input.detach(), dtype=torch.float32)
# model = model.full()
modeld = model.to(device)


for i in range(100):
    
    input_d = inputs.to(device)
    output = modeld(input_d)
    output = output.to('cpu')
    print(f'output : {output}')
end = time.time()-start
print(f'time : {end}sec')


/tmp/ipykernel_11250/21709948.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs = torch.tensor(input.detach(), dtype=torch.float32)


output : tensor([[-0.6506,  0.5699,  0.9909,  ..., -0.6971, -0.0040,  0.3011],
        [-0.5880,  0.3263,  0.6494,  ..., -0.4701,  0.1757,  0.2660],
        [-0.4764,  0.2062,  0.5736,  ..., -0.4387,  0.2734,  0.0986],
        [-0.6797,  0.2667,  0.9374,  ..., -0.5675, -0.0464,  0.2711]],
       grad_fn=<ToCopyBackward0>)
output : tensor([[-0.6506,  0.5699,  0.9909,  ..., -0.6971, -0.0040,  0.3011],
        [-0.5880,  0.3263,  0.6494,  ..., -0.4701,  0.1757,  0.2660],
        [-0.4764,  0.2062,  0.5736,  ..., -0.4387,  0.2734,  0.0986],
        [-0.6797,  0.2667,  0.9374,  ..., -0.5675, -0.0464,  0.2711]],
       grad_fn=<ToCopyBackward0>)
output : tensor([[-0.6506,  0.5699,  0.9909,  ..., -0.6971, -0.0040,  0.3011],
        [-0.5880,  0.3263,  0.6494,  ..., -0.4701,  0.1757,  0.2660],
        [-0.4764,  0.2062,  0.5736,  ..., -0.4387,  0.2734,  0.0986],
        [-0.6797,  0.2667,  0.9374,  ..., -0.5675, -0.0464,  0.2711]],
       grad_fn=<ToCopyBackward0>)
output : tensor([[-0.6506,  

In [8]:
output[:,18:25]

tensor([[3.5802, 1.8356, 1.5026, 5.7263, 4.8259, 4.7571, 1.8068],
        [3.0856, 1.6712, 1.2817, 5.1314, 3.9570, 4.0317, 1.9194],
        [3.5102, 1.8066, 1.5194, 5.2363, 4.4751, 4.3398, 2.1461],
        [3.3421, 1.6897, 1.4245, 5.4187, 4.6329, 4.4575, 1.9495]],
       grad_fn=<SliceBackward0>)

In [9]:
output[:,18:25]

tensor([[3.5802, 1.8356, 1.5026, 5.7263, 4.8259, 4.7571, 1.8068],
        [3.0856, 1.6712, 1.2817, 5.1314, 3.9570, 4.0317, 1.9194],
        [3.5102, 1.8066, 1.5194, 5.2363, 4.4751, 4.3398, 2.1461],
        [3.3421, 1.6897, 1.4245, 5.4187, 4.6329, 4.4575, 1.9495]],
       grad_fn=<SliceBackward0>)

In [20]:
ls = [4, 196, 12, 3, 64]
def mult(ls):
    res = 1
    for i in ls:
        res *= i
    return res
torch.arange(mult(ls)).reshape(ls).permute(3, 0, 2, 1, 4)[1,0,1,0,:]

tensor([320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333,
        334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347,
        348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361,
        362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375,
        376, 377, 378, 379, 380, 381, 382, 383])

In [33]:
Q = torch.arange(mult(ls)).reshape(ls).permute(3, 0, 2, 1, 4)[0].float()
K = torch.arange(mult(ls)).reshape(ls).permute(3, 0, 2, 1, 4)[1].float()
nn.LayerNorm((196,),)(torch.matmul(Q, K.permute(0, 1, 3, 2)))

tensor([[[[-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          ...,
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232]],

         [[-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          ...,
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232]],

         [[-1.7232, -1.7056, -1.6879,  ...,  1.6879,  1.7056,  1.7232],
          [-1.7232, -1.7056, -